# Why RNNs forget: the vanishing gradient, live

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudanum/MIA5150/blob/main/notebooks/sequential-models-rnn-gru-lstm/02_vanishing_gradients.ipynb)

A vanilla RNN can learn a task across 20 timesteps and fail *completely* across 200 — not because the task changed, but because the training signal does not survive the trip back in time. In this notebook we make that failure visible three ways: **analytically** (what happens when you multiply 60 Jacobian factors), **empirically** (Hochreiter's classic *adding problem* at T=20 versus T=200), and **directly** (plotting the gradient itself as it flows backward through the sequence). We finish by swapping the recurrent core for an LSTM and a GRU, and watch the same task become solvable again.

**Learning goals**

- Write backpropagation through time (BPTT) as a **product of per-step Jacobians** and explain why it must either vanish or explode.
- Quantify a **memory horizon**: the number of steps after which a shrinking gradient becomes useless.
- Implement the **adding problem**, the standard lie detector for long-range memory, and its mean-predictor baseline.
- Train a vanilla RNN at T=20 and T=200, observe the difference, and then **see the vanishing gradient itself** on a probe plot.
- Show that **LSTM and GRU** cores repair the failure, and understand why gradient clipping cannot.

## 1. The algebra of forgetting

An Elman (vanilla) RNN updates its hidden state as

$$h_t = \tanh\!\big(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b\big).$$

Training needs the gradient of the loss $L$ (computed at the end of the sequence) with respect to every parameter. Backpropagation through time chains backward over every step, and the Jacobian of the final state $h_T$ with respect to an earlier state $h_t$ is the **matrix product**

$$\frac{\partial h_T}{\partial h_t} \;=\; \prod_{k=t+1}^{T} \operatorname{diag}\!\big(\tanh'(a_k)\big)\, W_{hh}, \qquad a_k = W_{xh}\,x_k + W_{hh}\,h_{k-1} + b.$$

That is $T - t$ factors multiplied together. Each factor has norm at most $\|W_{hh}\|_2 \cdot \max_z|\tanh'(z)| \le \|W_{hh}\|_2$. So the per-step **gain** $g$ of this product decides everything:

- **$g < 1$ — the gradient vanishes.** The product shrinks like $g^{\,T-t}$, exponentially fast. The gradient that finally reaches the early timesteps is effectively zero, so the model *cannot learn* from distant inputs no matter how long we train.
- **$g > 1$ — the gradient explodes.** The product grows like $g^{\,T-t}$ and updates blow up. Annoying, but patchable: **gradient clipping** rescales any update whose norm exceeds a threshold.
- **$g \approx 1$ — the knife-edge** the field chased for years (orthogonal initializations, unitary RNNs) before gated architectures made it unnecessary.

Clipping works for explosions because a signal *exists*, it is merely too large. Nothing fixes vanishing inside this architecture, because after enough factors there is no signal left to rescue. The real fix (section 6) is architectural: give credit a path that does not have to survive a long product.

In [ ]:
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Reproducibility: same batches, same initial weights, same plots every run.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ---- Config: every knob you are invited to turn lives here ----
# Runtime scales roughly with STEPS * T. STEPS below is chosen so that, on a
# free Colab CPU, LSTM and GRU clearly beat the baseline at T=200 while a full
# "Run all" stays within a coffee break. For a quick end-to-end syntax check,
# STEPS = 60 runs the whole notebook in well under a minute.
HIDDEN = 32        # hidden units in every recurrent core (RNN, GRU, LSTM)
STEPS = 2500       # optimizer steps per training run
BATCH = 128        # sequences per training batch (fresh random data every step)
LR = 1e-3          # Adam learning rate
LOG_EVERY = 20     # record the loss every this many steps
EVAL_BATCH = 2048  # fresh sequences used for the final MSE estimate

## 2. Analytical demo: sixty multiplied Jacobians

Before training anything, watch the mechanism in isolation. We build a random $64 \times 64$ matrix $W_{hh}$ with a controlled spectral norm $\|W_{hh}\|_2 \in \{0.9,\ 1.0,\ 1.1\}$ — a scaled random *orthogonal* matrix, so every direction receives the same gain and the plot isolates the effect cleanly. Then we track $\big\|\prod^k W_{hh}\big\|_2$ as we multiply in one factor at a time, for $k = 1 \dots 60$.

Since $|\tanh'| \le 1$ can only shrink each factor further, this is already a *best case* for the vanilla RNN.

We also mark a **memory horizon**: the number of steps after which the signal has fallen below 1% of its starting magnitude. Beyond the horizon, the network is effectively unteachable about what happened — the information may still sit in the activations, but no gradient will ever reach it.

In [ ]:
N_DIM = 64       # size of the toy hidden state
K_MAX = 60       # how many Jacobian factors we multiply together
GAINS = (0.9, 1.0, 1.1)


def random_matrix_with_gain(n, gain, seed):
    """A random n x n matrix whose spectral norm is exactly `gain`.

    A random orthogonal matrix has all singular values equal to 1, so scaling
    it by `gain` gives a worst-case/best-case-equal gain in every direction.
    (A raw Gaussian matrix has a wide spread of singular values and would
    blur the story.)
    """
    g = torch.Generator().manual_seed(seed)
    q, _ = torch.linalg.qr(torch.randn(n, n, generator=g))
    return gain * q


plt.figure(figsize=(7.5, 4.5))
for gain in GAINS:
    W = random_matrix_with_gain(N_DIM, gain, seed=SEED)
    J = torch.eye(N_DIM)
    norms = []
    for k in range(1, K_MAX + 1):
        J = J @ W  # multiply in one more per-step Jacobian factor
        norms.append(torch.linalg.matrix_norm(J, ord=2).item())
    plt.semilogy(range(1, K_MAX + 1), norms, lw=2,
                 label=rf"$\|W_{{hh}}\|_2 = {gain}$")

# Memory horizon of the shrinking case: solve gain**k = THRESH for k.
THRESH = 1e-2
k_star = math.log(THRESH) / math.log(GAINS[0])
plt.axhline(1.0, color="k", lw=0.8, alpha=0.3)
plt.axhline(THRESH, color="gray", ls=":", lw=1)
plt.axvline(k_star, color="gray", ls=":", lw=1)
plt.annotate(f"memory horizon ~ {k_star:.0f} steps\n(gain {GAINS[0]}, 1% threshold)",
             xy=(k_star, THRESH), xytext=(2, 3e-4),
             arrowprops=dict(arrowstyle="->", color="gray"), fontsize=10)
plt.ylim(1e-4, 5e3)
plt.xlabel(r"number of multiplied factors  $k$")
plt.ylabel(r"$\left\|\prod^k W_{hh}\right\|_2$  (log scale)")
plt.title("Products of many factors: exponential decay, stalemate, or explosion")
plt.legend()
plt.tight_layout()
plt.show()

**What to notice**

- On a log scale the three curves are straight lines: multiplying Jacobians is *exponential* in the number of steps, never merely linear. A gain of 0.9 is not "10% lost per step" — it is the signal **halving roughly every 7 steps**.
- At gain exactly 1.0 the signal survives intact. This is why orthogonal and unitary initializations were such a big deal, and why gated architectures (which can hold a gradient path open) eventually replaced them.
- At 1.1 the product explodes to hundreds of times its start within 60 steps — which at least can be clipped. The 0.9 curve is the fatal one: past the dotted **memory horizon** (~44 steps here) less than 1% of the signal remains, and no optimizer setting can recover it.
- 60 factors is generous. The adding problem below uses T=200 — and a real RNN additionally multiplies by $\tanh' \le 1$ at every step, so it typically sits *below* these curves.

## 3. The adding problem: a lie detector for long memory

Hochreiter's adding problem is the minimal task that *requires* remembering across a long gap. Each sequence has $T$ timesteps; at each step the input is a pair $(x_t, m_t)$ with a value $x_t \sim U[0,1)$ and a marker bit $m_t \in \{0,1\}$ that equals 1 at **exactly two** random positions. The target is the sum of the two marked values:

$$y = \sum_{t:\, m_t = 1} x_t .$$

Trivial for a human; brutal for a vanilla RNN when $T$ is large. To learn, the gradient has to travel from the loss at the final step all the way back to wherever the markers landed — on average $T/2$ steps, sometimes nearly the full $T$.

**The baseline to beat.** The best *constant* predictor guesses the mean $\mathbb{E}[y] = 1$ and scores the variance, $\mathrm{Var}(y) = \tfrac{1}{6} \approx 0.167$ (sum of two independent $U[0,1)$ draws — independent of $T$). A model that cannot find the markers converges to exactly this. So the dashed line is not a straw man: **flatlining at the baseline *is* the failure mode** — it is what "the model learned nothing about positions" looks like.

In [ ]:
def make_batch(batch_size, T):
    """One batch for the adding problem.

    Returns
    -------
    inputs : (batch_size, T, 2) float tensor
        Channel 0 is the value x_t ~ U[0, 1); channel 1 is the marker m_t,
        with exactly two 1s per sequence.
    targets : (batch_size, 1) float tensor
        The sum of x_t at the two marked positions.
    """
    x = torch.rand(batch_size, T, 1)
    # Two distinct positions per sequence, uniformly: give every timestep a
    # uniform random key, keep the positions of the two smallest keys.
    keys = torch.rand(batch_size, T)
    idx = keys.topk(2, dim=1, largest=False).indices          # (batch, 2)
    m = torch.zeros(batch_size, T, 1)
    m.scatter_(1, idx.unsqueeze(-1), 1.0)
    inputs = torch.cat([x, m], dim=-1)
    targets = (x * m).sum(dim=1)
    return inputs, targets


# --- sanity check: look at one sequence before training anything ---
torch.manual_seed(SEED)
demo_x, demo_y = make_batch(1, 20)
marked = demo_x[0, :, 1].nonzero().flatten().tolist()
print(f"marked positions: {marked}")
print(f"x values there:   {[round(v, 4) for v in demo_x[0, marked, 0].tolist()]}")
print(f"target sum:       {demo_y.item():.4f}")

plt.figure(figsize=(8, 2.6))
seq = demo_x[0]
plt.plot(seq[:, 0].numpy(), ".-", lw=1, label="$x_t$ (value)")
plt.stem(marked, seq[marked, 0].numpy(), linefmt="r-", markerfmt="ro",
         basefmt=" ", label="marked positions")
plt.xlabel("timestep t")
plt.title(f"One adding-problem sequence: target = {demo_y.item():.3f}")
plt.legend()
plt.tight_layout()
plt.show()

# --- the baseline: MSE of the best CONSTANT predictor ---
# The best constant is the mean of the target distribution; its MSE is the
# variance. Estimate it on a large fresh sample.
with torch.no_grad():
    _, y_big = make_batch(8192, 20)
    BASELINE_MSE = ((y_big - y_big.mean()) ** 2).mean().item()
print(f"mean-predictor baseline MSE = {BASELINE_MSE:.4f}   (theory: 1/6 = {1/6:.4f})")

## 4. Same RNN, two sequence lengths

Now the empirical test. The model is deliberately tiny: 2 inputs, a single `tanh` recurrent core with 32 hidden units, and a linear readout of the **final** hidden state — so all memory has to squeeze through the recurrence. Training is *online*: a fresh random batch every step, which means the training loss is also an honest estimate of the test loss.

Everything is identical across the two runs — architecture, budget, optimizer, seed. Only $T$ changes:

- **T = 20:** the gradient crosses at most 20 Jacobian factors. Should be learnable.
- **T = 200:** the same architecture, ten times the distance. Watch the dashed line.

In [ ]:
class Adder(nn.Module):
    """2 -> HIDDEN recurrent core -> 1, reading out the FINAL hidden state."""

    def __init__(self, core="rnn", hidden=HIDDEN, nonlinearity="tanh",
                 forget_bias=1.0):
        super().__init__()
        cores = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}
        kwargs = {"nonlinearity": nonlinearity} if core == "rnn" else {}
        self.rnn = cores[core](2, hidden, batch_first=True, **kwargs)
        self.head = nn.Linear(hidden, 1)
        if core == "lstm" and forget_bias:
            # Start the LSTM's forget gate OPEN (Jozefowicz et al., 2015):
            # the gate is sigmoid(bias) ~ 0.88 instead of ~0.5, so the cell
            # begins life as an additive carry path. This is the standard
            # initialization for long-memory tasks -- and it is exactly the
            # "keep the gradient highway open" idea of this notebook. With
            # forget_bias=0.0 the LSTM stays near the baseline far longer
            # (the optimizer must first learn to open the gate).
            h = hidden
            with torch.no_grad():
                self.rnn.bias_ih_l0[h:2 * h].fill_(forget_bias)
                self.rnn.bias_hh_l0[h:2 * h].fill_(forget_bias)

    def forward(self, inputs):
        out, _ = self.rnn(inputs)      # out: (batch, T, hidden)
        return self.head(out[:, -1])   # only the last step's summary is read


def train_adding(core, T, steps=STEPS, nonlinearity="tanh", grad_clip=None,
                 seed=SEED, **model_kwargs):
    """Train one model on the adding problem with fresh data every step.

    Returns (steps, losses, final_mse, model). `grad_clip` is here so the
    exercises can verify that clipping does not rescue a vanishing gradient.
    """
    torch.manual_seed(seed)
    model = Adder(core, nonlinearity=nonlinearity, **model_kwargs)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    mse = nn.MSELoss()
    hist_steps, hist_loss = [], []
    t0 = time.time()
    for step in range(1, steps + 1):
        inputs, targets = make_batch(BATCH, T)
        loss = mse(model(inputs), targets)
        opt.zero_grad()
        loss.backward()
        if grad_clip is not None:  # rescues explosions; cannot create signal
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        opt.step()
        if step == 1 or step % LOG_EVERY == 0:
            hist_steps.append(step)
            hist_loss.append(loss.item())
    with torch.no_grad():
        inputs, targets = make_batch(EVAL_BATCH, T)
        final_mse = mse(model(inputs), targets).item()
    print(f"{core.upper():4s}  T={T:3d}:  final eval MSE = {final_mse:.5f}"
          f"   (baseline = {BASELINE_MSE:.4f})   [{time.time() - t0:.0f}s]")
    return np.array(hist_steps), np.array(hist_loss), final_mse, model

In [ ]:
s20, l20, mse20, model20 = train_adding("rnn", 20)
s200, l200, mse200, model200 = train_adding("rnn", 200)

plt.figure(figsize=(7.5, 4.5))
plt.axhline(BASELINE_MSE, ls="--", c="gray",
            label=f"mean-predictor baseline ~ {BASELINE_MSE:.3f}")
plt.plot(s20, l20, lw=2, label="RNN, T=20")
plt.plot(s200, l200, lw=2, label="RNN, T=200")
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("MSE (log scale)")
plt.title("Same RNN, same budget: short sequences get learned, long ones don't")
plt.legend()
plt.tight_layout()
plt.show()

**What to notice**

- At T=20 the loss drops well below the dashed baseline: the RNN found the markers.
- At T=200 the curve sits on (or barely under) the baseline for the entire run. This is not ordinary underfitting, and more steps will not save it: the training signal about the marked positions never arrives.
- Exactly what sections 1–2 predicted: a product of ~200 Jacobian factors with gain below 1 is numerically zero long before it reaches the markers.

## 5. The smoking gun: plotting the gradient itself

Claims about vanishing gradients usually stay abstract. Here we make it literal: take one batch at T=200, run a **single** backward pass, and read off $\lvert \partial L / \partial x_t \rvert$ at every timestep. If credit assignment worked, the loss would "care about" early inputs roughly as much as late ones.

We probe two models: a **fresh, untrained RNN** — to show the effect is architectural, present before any learning — and the **RNN that just flatlined** at T=200.

In [ ]:
def grad_probe(model, T, batch=256):
    """Mean |dL/dx_t| at every timestep from ONE backward pass."""
    torch.manual_seed(SEED)
    inputs, targets = make_batch(batch, T)
    inputs.requires_grad_(True)
    loss = nn.MSELoss()(model(inputs), targets)
    loss.backward()
    # average over the batch, keep the value channel (channel 0)
    g = inputs.grad[..., 0].abs().mean(dim=0)
    return g.numpy()


T_PROBE = 200
torch.manual_seed(SEED)
fresh_rnn = Adder("rnn")                  # untrained: the problem is structural
g_fresh = grad_probe(fresh_rnn, T_PROBE)
g_trained = grad_probe(model200, T_PROBE)  # the RNN that flatlined above

plt.figure(figsize=(7.5, 4.5))
plt.semilogy(g_fresh + 1e-30, lw=2, label="fresh RNN (untrained)")
plt.semilogy(g_trained + 1e-30, lw=2, label="RNN trained at T=200")
plt.xlabel("timestep t")
plt.ylabel(r"mean $|\partial L / \partial x_t|$  (log scale)")
plt.title(f"The gradient dies on the way back (T={T_PROBE})")
plt.legend()
plt.tight_layout()
plt.show()

ratio = g_trained[-1] / max(g_trained[0], 1e-30)
print(f"trained RNN: gradient at t={T_PROBE - 1} is ~{ratio:.1e}x "
      f"larger than at t=0")

**What to notice**

- On the log scale, gradient magnitude falls off roughly linearly as you move left (back in time): that is exponential decay made visible. The gradient at $t=0$ is orders of magnitude smaller than at $t=T-1$ — early inputs are simply invisible to the loss.
- The untrained RNN shows the same pattern: nothing went wrong *during* training. The geometry of the architecture guarantees it.
- Where a curve flattens at the bottom of the plot, the gradient has underflowed float32 — the signal is not just small, it is *numerically absent*.

## 6. The architectural fix: LSTM and GRU

LSTM and GRU were invented precisely to break the product. Their gated updates give credit an **additive** path: information can be carried forward (and gradients backward) across hundreds of steps without being multiplied by a fresh $W_{hh}\,\operatorname{diag}(\tanh')$ at every step. The gates learn *when* to keep that path open.

One honest implementation detail: we initialize the LSTM with its **forget gate open** (bias 1, the classic trick of Jozefowicz et al., 2015 — see the comment in `Adder`). That simply starts the carry path in the "open" position instead of making the optimizer discover it; with the default initialization the LSTM stays near the baseline far longer than a live demo allows. Everything else — task (T=200), budget, trainer, seed — is unchanged. Only the recurrent core changes.

In [ ]:
s_l, l_l, mse_l, model_l = train_adding("lstm", 200)
s_g, l_g, mse_g, model_g = train_adding("gru", 200)

plt.figure(figsize=(7.5, 4.5))
plt.axhline(BASELINE_MSE, ls="--", c="gray",
            label=f"mean-predictor baseline ~ {BASELINE_MSE:.3f}")
plt.plot(s200, l200, lw=2, label="RNN")
plt.plot(s_l, l_l, lw=2, label="LSTM")
plt.plot(s_g, l_g, lw=2, label="GRU")
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("MSE (log scale)")
plt.title("Same task (T=200), same budget, different core")
plt.legend()
plt.tight_layout()
plt.show()

**What to notice**

- LSTM and GRU both leave the baseline and keep falling: with the gradient path repaired, T=200 is no longer a long distance. (Here GRU happens to take off earlier than LSTM — a seed-and-init effect, not a law of nature.)
- The vanilla RNN never moves. Nothing about the data, budget, or optimizer changed — only the architecture's gradient geometry.
- Skeptical about the forget-gate initialization? Rerun with `train_adding("lstm", 200, forget_bias=0.0)`: within this budget the curve stays pinned near the baseline — the optimizer first has to *discover* the open-carry configuration, which takes much longer than 2500 steps. The vanilla RNN has no such trick available at all: there is no gate to open.
- Could gradient clipping rescue the RNN instead? No. Clipping rescales gradients that are too *large*; at T=200 the RNN's gradients are too *small*. There is nothing to rescale. You will verify this yourself below.

## Try it yourself

1. **Map the failure boundary.** Sweep `T in {20, 50, 100, 200, 400}` for each core (`"rnn"`, `"lstm"`, `"gru"`) with `train_adding`, and plot final eval MSE vs T — one curve per core, log y-axis, dashed baseline. Where does the vanilla RNN cross the baseline? Do LSTM/GRU degrade at all by T=400? (That is 15 training runs — drop `STEPS` to ~400, or sweep one core at a time, if you are impatient.)

2. **ReLU instead of tanh.** Rerun the RNN arms with `train_adding("rnn", T, nonlinearity="relu")`. Since $\mathrm{ReLU}' \in \{0, 1\}$ never squashes below 1, does the T=200 picture improve? And what *new* failure appears instead — watch the printed losses for `nan` and explain why it happens.

3. **Clipping cannot rescue vanishing.** Run `train_adding("rnn", 200, grad_clip=1.0)` and compare against the unclipped T=200 curve. In one sentence: why is clipping a one-sided fix?

4. **Move the horizon.** In the section-2 demo, change the gains to `(0.95, 0.99, 1.01)` and recompute the 1% memory horizon for the two shrinking cases. Roughly how does the horizon scale as the gain approaches 1? (Hint: $\ln g \approx g - 1$ for $g$ near 1.)

## Recap

- BPTT multiplies $\sim T$ per-step Jacobians $W_{hh}^\top \operatorname{diag}(\tanh')$; with per-step gain $g$, credit decays (or explodes) like $g^{\,T-t}$. Exponential means: the **number of steps**, not the task difficulty, decides what is learnable.
- The **memory horizon** — steps until the signal falls below 1% — is $\ln(0.01)/\ln g$: only ~44 steps at $g = 0.9$, and a real RNN's $\tanh'$ factors push it lower still.
- On the adding problem, a vanilla RNN solves T=20 but flatlines at the mean-predictor baseline for T=200, and a direct probe shows $|\partial L / \partial x_t|$ collapsing toward $t = 0$: the vanishing gradient, live.
- Gradient clipping cures *explosion* only. *Vanishing* is fixed architecturally: LSTM/GRU gates provide additive gradient highways, and the same T=200 task becomes learnable.
- Next in the module: we open up the LSTM and GRU gates themselves and watch what they choose to remember.